# Order-book observations: what market state can I reconstruct?

**Research question:** what visible liquidity can I observe in a bounded recording,
can I reconstruct it from saved inputs, and when do gaps prevent me trusting it?

This notebook explores **Bitfinex and Coinbase individual visible orders (L3)** and
**OKX aggregate price levels (L2)**. L2 says how much is displayed at a price; L3 also
retains visible order identities. Neither alone establishes hidden liquidity,
queue priority or whether a hypothetical order would fill. USD and USDT remain
separate quote currencies.

| Reading route | Research outcome |
| --- | --- |
| 1–4: choose and obtain observations | Understand the selected instruments, access status and bounded recordings. |
| 5–8: inspect and challenge a recording | Read coverage, clocks and saved reconstruction; observe a gap and a synthetic order lifecycle. |
| 9–11: compare historical and live evidence | Inspect an explicit OKX historical prefix and compare only what each source actually supplies. |

**Before Run All:** this activity contacts sources, makes new live recordings and
acquires the configured OKX historical archive (about 104 MiB in the recorded run).
Coinbase live collection runs only with eligible local Exchange credentials;
otherwise its blocker is shown. Each recording ends at its configured limits.
Reading saved outputs does not start collection. Restart and execute all cells in
order when intentionally collecting; later sections use the earlier saved inputs.

Expand collapsed source cells to inspect the code. For trade-history research,
start with [the Binance historical notebook](research-foundations.ipynb).
[Researcher's guide](README.md) · [Commands](notebook-operations.md) ·
[Collector details](../docs/data/l3-collectors.md) · [OKX details](../docs/data/okx-market-data.md).

Presentation updated 2026-09-16. Existing source code and displayed outputs are
preserved from their recorded execution; no new live capture is claimed by this edit.

## 1. Which source and implementation will this study use?

The setup selects the maintained venue configurations and identifies the installed
code and lockfile. **Read:** these are the inputs for this run, not a comparison of
venue quality. Configurations bound recording duration, message/storage size and
visible state. [Setup help](notebook-operations.md#before-executing).

In [1]:
import hashlib
import json
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

import alpha_research
import pandas as pd

from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.live.transport import VenueTransport
from alpha_research.data.venues.coinbase.auth import CoinbaseAccessError, CoinbaseCredentials
from alpha_research.data.venues.coinbase.book import CoinbaseBook
from alpha_research.data.venues.okx.book import OKXBook
from alpha_research.data.venues.okx.history_config import OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.data.venues.okx.history_transport import OKXHistoryTransport
from alpha_research.pipelines.acquisition import acquire_history
from alpha_research.pipelines.live import InstalledIdentity, collect
from alpha_research.serialization import DomainCodec

root = Path.cwd()
assert (root / "configs/live").is_dir(), "Run from the repository root"
codec = DomainCodec()
implementation = InstalledIdentity.sha256()
bitfinex = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
coinbase = CaptureConfig.read(root / "configs/live/coinbase-btc-usd.json")
okx = CaptureConfig.read(root / "configs/live/okx-btc-usdt.json")
okx_history = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
assert okx.instrument_id == okx_history.instrument_id
pd.DataFrame([{"python": sys.version.split()[0], "package": alpha_research.__file__,
               "implementation_sha256": implementation,
               "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest()}])

,python,package,implementation_sha256,lock_sha256
0,3.14.7,/private/tmp/crypto-l3-verification/consumer/l...,e00f84a13948b3cf89ed8308a83e8ca2a5ded54fb83633...,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a...


## 2. Which instruments and types of liquidity can I observe?

**Read:** the discovery table describes current source availability and the selected
native instruments. It does not prove historical listing membership or entitlement
to authenticated Coinbase L3. Bitfinex is public; Coinbase needs eligible local
Exchange credentials. The selected OKX source is public `books`, with 400 aggregate
price levels per side and its configured nominal update cadence.

Keep the representation and quote currency attached to any comparison: aggregate
price levels are not individual orders, and BTC/USD is not BTC/USDT.

In [2]:
discovery = []
for config in (bitfinex, coinbase, okx):
    payload = VenueTransport(config).discover()
    discovery.append({"venue": config.venue, "representation": "L2" if config.venue == "okx" else "L3", "native_symbol": config.symbol,
                      "instrument_id": str(config.instrument_id), "channel": config.channel,
                      "duration_s": config.duration_seconds, "message_limit": config.max_messages,
                      "journal_bytes": config.max_bytes, "depth": config.depth,
                      "discovery_sha256": hashlib.sha256(payload).hexdigest()})
pd.DataFrame(discovery)

,venue,representation,native_symbol,instrument_id,channel,duration_s,message_limit,journal_bytes,depth,discovery_sha256
0,bitfinex,L3,tBTCUSD,c87d983f-e7ec-4e64-8d54-9f2be67a64b8,R0,20,10000,32000000,25,ad5c2c16d99b75060bdfcb43aa70faa233931623d8b953...
1,coinbase,L3,BTC-USD,218142de-d549-43bb-85ba-a174fd4f798f,level3,20,10000,32000000,0,d2bfd4363396ca895bed3a49a2e3f265c7c043efbb50b0...
2,okx,L2,BTC-USDT,4476df8e-6342-4417-839e-e41b88f41446,books,20,10000,32000000,400,5868aba36f44605b113b2602986f4e2b2e08c55b6bb638...


## 3. What happened during this bounded live sample?

The next cell records Bitfinex and OKX, saves the incoming messages and reconstructs
the visible book. **Read:** the returned status, message count, usable interval and
gap reasons describe this particular attempt. A later successful recovery does not
erase a preceding invalid interval.

Each call stops at configured limits and retains its own recording directory.
Connection/control messages and local receipt times stay with market messages,
so later inspection can explain why a book state was or was not usable.

In [3]:
bitfinex_result = await collect(bitfinex, root / "data/bitfinex",
                                implementation_identity=implementation)
assert bitfinex_result["replay_parity"]
assert bitfinex_result["synchronized"] and not bitfinex_result["gaps"], bitfinex_result
okx_result = await collect(okx, root / "data/okx/live", implementation_identity=implementation)
assert okx_result["replay_parity"]
assert okx_result["synchronized"] and not okx_result["gaps"], okx_result
assert okx_result["orders"] is None and okx_result["representation"] == "L2"
{"bitfinex": bitfinex_result, "okx": okx_result}


{'bitfinex': {'venue': 'bitfinex',
  'symbol': 'tBTCUSD',
  'instrument_id': 'c87d983f-e7ec-4e64-8d54-9f2be67a64b8',
  'channel': 'R0',
  'started_ns': 1789531801467557000,
  'finished_ns': 1789531821633034000,
  'messages': 382,
  'updates': 376,
  'snapshots': 1,
  'heartbeats': 2,
  'depth': 25,
  'orders': 50,
  'max_orders': 100000,
  'gaps': [],
  'synchronized': True,
  'ended': True,
  'source_sequence': 379,
  'last_source_time': None,
  'book_identity': '61b8ea219edc4f6bece997e5f32775d448294f98455d5ab9da54cdab65ff70f0',
  'trace_identity': '49dfbe292f56ecab931b1b8b387bff2a134a24ab6c186015081f62188c393cc7',
  'epochs': 1,
  'request_identity': '7e1062fd490d20edcc3eb0cfb59a9b5fabc83207393de526d37652ecc71c7d97',
  'saved_input_identity': 'bdad61d34dd3311d444701e6b71b31cedc991b8d5052629f2e71364603cebb0b',
  'saved_bytes': 53192,
  'capture_directory': '/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/613b3e58-f654-4420-87b3-601554977213',
  'sto

## 4. Do I actually have Coinbase observations for this comparison?

**Read:** distinguish a real completed Coinbase recording from an access blocker.
Missing or invalid local credentials are reported before connection. If eligible
credentials are available, this cell makes a bounded recording; their values are
never displayed. Follow the [collector guide](../docs/data/l3-collectors.md) for local
setup. A successful synthetic exercise later is not a substitute for this access.

In [4]:
coinbase_result = None
try:
    exchange_credentials = CoinbaseCredentials.read(root / "secrets/coinbase.secrets")
except CoinbaseAccessError as error:
    coinbase_access = {"venue": "coinbase", "status": "access_blocked", "reason": str(error)}
else:
    coinbase_result = await collect(coinbase, root / "data/coinbase",
                                   implementation_identity=implementation,
                                   credentials=exchange_credentials)
    del exchange_credentials
    coinbase_access = {"venue": "coinbase", "status": "captured",
                       "synchronized": coinbase_result["synchronized"],
                       "gaps": coinbase_result["gaps"]}
coinbase_access

{'venue': 'coinbase',
 'status': 'access_blocked',
 'reason': 'Fill COINBASE_API_KEY, COINBASE_API_SECRET (base64), and COINBASE_API_PASSPHRASE in the Exchange secrets file'}

## 5. Which interval and part of the book does my sample cover?

**Read:** counts and identities refer to the actual recording. Local receipt clocks
are UTC observations by this application, not invented exchange timestamps.
Bitfinex has no source clock under the selected settings; its depth-25 window is a
bounded visible sample. OKX reports aggregate levels, with individual `orders=None`.

Do not compare the order and level counts as the same quantity. An unknown clock
or count remains unknown, rather than being replaced with zero.

In [5]:
results = [bitfinex_result, okx_result] + ([] if coinbase_result is None else [coinbase_result])
rows = []
for result in results:
    row = {name: result[name] for name in ("venue", "symbol", "messages", "snapshots", "updates",
                                          "orders", "depth", "gaps", "saved_input_identity")}
    row["levels"] = result.get("levels")
    row["representation"] = "L2" if result["venue"] == "okx" else "L3"
    for name in ("started_ns", "finished_ns"):
        row[name.replace("_ns", "_utc")] = (
            datetime(1970, 1, 1, tzinfo=UTC) + timedelta(microseconds=result[name] // 1000)
        ).isoformat()
    rows.append(row)
pd.DataFrame(rows)

,venue,symbol,messages,snapshots,updates,orders,depth,gaps,saved_input_identity,levels,representation,started_utc,finished_utc
0,bitfinex,tBTCUSD,382,1,376,50.0,25,[],bdad61d34dd3311d444701e6b71b31cedc991b8d505262...,NaN,L3,2026-09-16T04:10:01.467557+00:00,2026-09-16T04:10:21.633034+00:00
1,okx,BTC-USDT,186,1,184,NaN,400,[],c3df4896f45eadaab1810dbbe8c9254bcb9e0feb1c487d...,800.0,L2,2026-09-16T04:10:21.644895+00:00,2026-09-16T04:10:41.732980+00:00


## 6. Can I trace the reconstruction back to the saved messages?

The next cell verifies saved bytes and inspects the receipt record and numeric
types used by the same decoder as collection. **Read:** the fingerprint links the
reconstruction to an original file; exact numeric representation preserves prices
and sizes. Bulk feed data remain in the ignored venue directory.

For OKX, an absolute size replaces the prior size at a price and zero deletes that
level. Live sequence links help detect invalid transitions. These mechanics support
reconstruction; they are not assumptions about a strategy's fills.

In [6]:
capture_root = Path(bitfinex_result["capture_directory"])
request = codec.decode_json((capture_root / "request.json").read_bytes(), CaptureRequest)
journal = CaptureJournal(capture_root, request)
first_frame = next(record for record in journal.records() if record.kind == "frame")
offline = CaptureReplay.read(capture_root)
example_order = next(iter(offline.book.orders.values()))
okx_root = Path(okx_result["capture_directory"])
okx_offline = CaptureReplay.read(okx_root)
assert isinstance(okx_offline.book, OKXBook)
okx_price, okx_level = next(iter(okx_offline.book.bids.items()))
pd.DataFrame([
    {"venue": "bitfinex", "epoch": first_frame.epoch, "receipt_ns": first_frame.received_ns,
     "raw_frame_bytes": len(first_frame.raw),
     "raw_frame_sha256": hashlib.sha256(first_frame.raw).hexdigest(),
     "order_id_type": type(example_order.order_id).__name__,
     "price_type": type(example_order.price).__name__, "size_type": type(example_order.size).__name__,
     "source_time": offline.book.source_time},
    {"venue": "okx", "order_id_type": None, "price_type": type(okx_price).__name__,
     "size_type": type(okx_level.size).__name__, "reported_orders_at_price": okx_level.order_count,
     "source_time": okx_offline.book.source_time, "source_time_unit": "unix_milliseconds"},
])


,venue,epoch,receipt_ns,raw_frame_bytes,raw_frame_sha256,order_id_type,price_type,size_type,source_time,reported_orders_at_price,source_time_unit
0,bitfinex,0.0,1.789532e+18,102.0,06e9379502b7c6f703950eeaeed80bf278a966581d0882...,str,Decimal,Decimal,NaN,NaN,NaN
1,okx,NaN,NaN,NaN,NaN,NaN,Decimal,Decimal,1789531841501,1.0,unix_milliseconds


## 7. Does the reconstructed book survive a data gap?

First compare the saved reconstruction with the recording's sequence of book
states. **Read:** matching results establish repeatable processing of these saved
messages. Keep the actual files, request and manifest; a matching fingerprint
cannot recover a lost file.

**Synthetic challenge:** append a disconnect to a copy of the observed prefix.
A heartbeat alone must not make the book usable again: a fresh subscription and
snapshot are required. The real saved input is unchanged. A recovered book does
not retroactively fill the missing interval.

In [7]:
assert offline.summary()["trace_identity"] == bitfinex_result["trace_identity"]
assert offline.summary()["book_identity"] == bitfinex_result["book_identity"]
assert offline.messages == bitfinex_result["messages"]
assert okx_offline.summary()["trace_identity"] == okx_result["trace_identity"]
assert okx_offline.summary()["book_identity"] == okx_result["book_identity"]
exercise = CaptureReplay(bitfinex)
for record in journal.records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(CaptureRecord(kind="gap", epoch=exercise.epoch,
                             received_ns=exercise.finished_ns, reason="synthetic_disconnect"))
assert not exercise.book.ready
{"offline_parity": True, "okx_offline_parity": True, "verified_messages": offline.messages,
 "trace_sha256": offline.summary()["trace_identity"],
 "synthetic_gap": exercise.gaps[-1], "usable_after_gap": exercise.book.ready}

{'offline_parity': True,
 'okx_offline_parity': True,
 'verified_messages': 382,
 'trace_sha256': '49dfbe292f56ecab931b1b8b387bff2a134a24ab6c186015081f62188c393cc7',
 'synthetic_gap': {'epoch': 0,
  'received_ns': 1789531821632238000,
  'reason': 'synthetic_disconnect'},
 'usable_after_gap': False}

## 8. How does a visible order change after a match?

This independent **synthetic** Coinbase example starts with a small order snapshot,
bridges the next sequence and applies a maker match that reduces resting size.
**Read:** watch the visible order lifecycle and distinguish non-resting messages
from actual visible opens.

The exercise checks the reconstruction behavior even when Coinbase access is
blocked. It establishes neither real access nor queue priority or execution quality.

In [8]:
coinbase_book = CoinbaseBook(coinbase)
coinbase_book.apply((root / "tests/fixtures/l3/coinbase-schema.json").read_bytes())
coinbase_book.apply(b'{"type":"subscriptions","channels":[{"name":"level3","product_ids":["BTC-USD"]}]}')
coinbase_book.apply(b'["noop","BTC-USD","11",null]')
coinbase_book.apply((root / "tests/fixtures/l3/coinbase-snapshot.json").read_bytes(), snapshot=True)
coinbase_book.apply(b'["match","BTC-USD","12","maker","taker","100","0.1",null]')
assert str(coinbase_book.orders["maker"].size) == "0.200000000000000001"
assert coinbase_book.ready
{"evidence": "synthetic", "sequence": coinbase_book.sequence,
 "remaining_size": str(coinbase_book.orders["maker"].size),
 "orders": len(coinbase_book.orders), "source_time": coinbase_book.source_time,
 "authenticated_live_evidence": coinbase_access["status"]}

{'evidence': 'synthetic',
 'sequence': 12,
 'remaining_size': '0.200000000000000001',
 'orders': 1,
 'source_time': None,
 'authenticated_live_evidence': 'access_blocked'}

## 9. What does the selected historical OKX archive contain?

The maintained request selects **8 September 2026**, one native instrument and the
400-level L2 archive. The next cell discovers the source reference, downloads the
whole compressed day, verifies it and saves the original bytes with its request.
**Read:** a completed download establishes a retained archive, not full-day book
continuity or a validated research dataset.

The recorded run downloaded about 104 MiB. Configured caps are 160 MB compressed
and 1 GB expanded. An incomplete attempt has no completion manifest and a retry
restarts the download within finite limits. [Acquisition details](../docs/data/okx-market-data.md).

In [9]:
index = OKXHistoryTransport(okx_history).index()
url = okx_history.select_url(index)
print({"utc_day": okx_history.day.isoformat(), "module": 4, "url": url,
       "compressed_limit": okx_history.max_bytes,
       "expanded_limit": okx_history.max_expanded_bytes,
       "replay_record_limit": okx_history.max_records})
history_result = acquire_history(okx_history, root / "data/okx/history",
                                 implementation_identity=implementation)
assert history_result["state_available"]
history_result


{'utc_day': '2026-09-08', 'module': 4, 'url': 'https://static.okx.com/cdn/okx/match/orderbook/pro/L2/400lv/daily/20260908/BTC-USDT-L2orderbook-400lv-2026-09-08.tar.gz', 'compressed_limit': 160000000, 'expanded_limit': 1000000000, 'replay_record_limit': 10000}


{'venue': 'okx',
 'symbol': 'BTC-USDT',
 'instrument_id': '4476df8e-6342-4417-839e-e41b88f41446',
 'representation': 'L2',
 'mode': 'historical',
 'depth': 400,
 'orders': None,
 'records': 10000,
 'snapshots': 1,
 'updates': 9999,
 'levels': 800,
 'source_sequence': None,
 'first_source_ms': '1788825600004',
 'last_source_ms': '1788825824624',
 'record_receipt_time': None,
 'continuity': 'unproven_no_source_sequences',
 'gaps': None,
 'state_available': True,
 'expanded_bytes': 764149760,
 'stop_reason': 'record_limit',
 'entire_archive_replayed': False,
 'book_identity': '98463d3904ce63326fb0e2a9bab95b927a6068289aa1989cebe66e18fabdf315',
 'trace_identity': 'e8c58ef9bc585227184ea43f5ced9ded45ba451ea91b0251544056ef5c5e6811',
 'saved_input_identity': 'fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676',
 'saved_bytes': 109423187,
 'request_identity': '503f9033527bc16073e080159a647ed06814e89345d4e0e71db755b780744469',
 'index_identity': 'c02a0194941bfd72bf14d5609381c8fe791a

## 10. How much historical book state did I actually reconstruct?

The whole compressed archive is retained, but this notebook processes only the
first **10,000 records**. **Read:** the displayed source interval describes that
prefix, not a full day of interpreted data.

The historical records provide price, size and order-count fields but no original
receipt timestamps or sequence IDs. Therefore `gaps=None` means continuity is
unknown; it does not mean zero gaps. These records cannot automatically repair
live gaps, and archive sampling is not assumed equivalent to the live feed.

In [10]:
offline_history = OKXArchiveStore(Path(history_result["archive_directory"])).replay()
assert offline_history["trace_identity"] == history_result["trace_identity"]
assert offline_history["book_identity"] == history_result["book_identity"]
assert offline_history["records"] == history_result["records"]
assert offline_history["source_sequence"] is None and offline_history["gaps"] is None
pd.DataFrame([{key: offline_history[key] for key in (
    "venue", "symbol", "records", "snapshots", "updates", "levels", "depth",
    "first_source_ms", "last_source_ms", "continuity", "entire_archive_replayed",
    "stop_reason", "saved_input_identity", "trace_identity")}])


,venue,symbol,records,snapshots,updates,levels,depth,first_source_ms,last_source_ms,continuity,entire_archive_replayed,stop_reason,saved_input_identity,trace_identity
0,okx,BTC-USDT,10000,1,9999,800,400,1788825600004,1788825824624,unproven_no_source_sequences,False,record_limit,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...,e8c58ef9bc585227184ea43f5ced9ded45ba451ea91b02...


## 11. What can I compare across these sources?

Read source representation, observed interval, clocks, saved identity and coverage
together. Receipt intervals refer to live recordings; source intervals refer to
the processed historical prefix. Coinbase's actual access outcome remains in
section 4. Differences in these columns limit comparisons of apparent activity
or liquidity across sources.

In [11]:
epoch = datetime(1970, 1, 1, tzinfo=UTC)
evidence = []
for result in results:
    evidence.append({"venue": result["venue"], "symbol": result["symbol"], "mode": "live",
        "representation": "L2" if result["venue"] == "okx" else "L3",
        "start_utc": (epoch + timedelta(microseconds=result["started_ns"] // 1000)).isoformat(),
        "end_utc": (epoch + timedelta(microseconds=result["finished_ns"] // 1000)).isoformat(),
        "clock": "receipt", "records": result["messages"], "orders": result["orders"],
        "levels": result.get("levels"), "depth": result["depth"], "gaps": result["gaps"],
        "saved_input_sha256": result["saved_input_identity"]})
evidence.append({"venue": "okx", "symbol": okx_history.symbol, "mode": "historical prefix",
    "representation": "L2", "clock": "source", "records": offline_history["records"],
    "start_utc": (epoch + timedelta(milliseconds=int(offline_history["first_source_ms"]))).isoformat(),
    "end_utc": (epoch + timedelta(milliseconds=int(offline_history["last_source_ms"]))).isoformat(),
    "orders": None, "levels": offline_history["levels"], "depth": 400, "gaps": None,
    "saved_input_sha256": offline_history["saved_input_identity"]})
pd.DataFrame(evidence)


,venue,symbol,mode,representation,start_utc,end_utc,clock,records,orders,levels,depth,gaps,saved_input_sha256
0,bitfinex,tBTCUSD,live,L3,2026-09-16T04:10:01.467557+00:00,2026-09-16T04:10:21.633034+00:00,receipt,382,50.0,NaN,25,[],bdad61d34dd3311d444701e6b71b31cedc991b8d505262...
1,okx,BTC-USDT,live,L2,2026-09-16T04:10:21.644895+00:00,2026-09-16T04:10:41.732980+00:00,receipt,186,NaN,800.0,400,[],c3df4896f45eadaab1810dbbe8c9254bcb9e0feb1c487d...
2,okx,BTC-USDT,historical prefix,L2,2026-09-08T00:00:00.004000+00:00,2026-09-08T00:03:44.624000+00:00,source,10000,NaN,800.0,400,None,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...


## What this study establishes, and where to stop

The recorded outputs show bounded source observations and whether saved
reconstruction matched them. The synthetic cases explain why a gap invalidates a
book and how an order lifecycle is processed. The meaning of each result remains
limited to its source, access outcome, representation and processed interval.

For OKX history, byte integrity and prefix reconstruction are separate from
continuity, which remains unproven without source sequences. The selected public
`books` representation excludes ELP/RPI liquidity. No archive/live sampling
equivalence, hidden liquidity, queue position or hypothetical fill is inferred.

Shared dataset publication, generalized replay scheduling and execution modeling
remain with their roadmap owners. The next useful research question is whether
these observations, intervals and limitations are sufficient for the specific
liquidity measurement you want to investigate. Preserve this attempt before
changing that question, and close the execution kernel when done.